In [ ]:
!wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt -O tinyshakespeare.txt
!head -100 tinyshakespeare.txt

In [ ]:
from torch import nn

class Encoder:

    def __init__(self, path):
        with open(path) as f:
            lines = f.readlines()

        text = "\n".join(lines)

        index = 1
        self.dictionary = {"⒨": 0}
        for char in text:
            if char in self.dictionary:
                continue

            self.dictionary[char] = index
            index += 1

    def encode(self, text):
        return [self.dictionary[char] for char in text]

    def decode(self, encoded) -> list[int]:
        inv_dict = {v: k for k, v in self.dictionary.items()}
        return [inv_dict[encoding] for encoding in encoded]

    def vocab(self):
        return self.dictionary.keys()

encoder = Encoder("tinyshakespeare.txt")
encoded = encoder.encode("Hello World!")
print(encoded)
decoded = encoder.decode(encoded)
print(decoded)


In [ ]:
from torch import nn, Tensor, softmax, randn

class Transformer(nn.Module):

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.encoder = Encoder("tinyshakespeare.txt")

        vocab_size = len(self.encoder.vocab())
        max_seq_len = 1024
        embed_dim = 256
        hidden_dim = 4 * embed_dim
        num_layers = 16

        self.token_emb = nn.Parameter(randn(vocab_size, embed_dim))
        self.pos_emb = nn.Parameter(randn(max_seq_len, embed_dim))

        self.layers = nn.ModuleList([
            nn.ModuleList([
                nn.LayerNorm(embed_dim),
                nn.Linear(embed_dim, embed_dim), #W_Q
                nn.Linear(embed_dim, embed_dim), #W_K
                nn.Linear(embed_dim, embed_dim), #W_V

                #FFN
                nn.LayerNorm(embed_dim),
                nn.Linear(embed_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, embed_dim)])
            for _ in range(num_layers)
        ])

        self.output = nn.Linear(embed_dim, vocab_size)

    def embedding(self, x: Tensor, pos_ids: Tensor | None = None):
        if pos_ids is None:
            pos_ids = torch.arange(x.shape[-1], device=x.device)
        return self.token_emb[x] + self.pos_emb[pos_ids]

    def attention(self, Q: Tensor, K: Tensor, V: Tensor):
        d_k = K.shape[-1]
        scores = Q @ K.transpose(-2, -1) / (d_k ** 0.5)
        seq_len = scores.shape[-1]
        causal = torch.tril(torch.ones(seq_len, seq_len, dtype=torch.bool, device=scores.device))
        scores = scores.masked_fill(~causal, -float("inf"))
        return softmax(scores, dim=-1) @ V

    def forward(self, x: Tensor, pos_ids: Tensor | None = None) -> Tensor:
        x = self.embedding(x, pos_ids)
        for ln1, W_Q, W_K, W_V, ln2, linear1, relu, linear2 in self.layers:
            h = ln1(x)
            x = x + self.attention(W_Q(h), W_K(h), W_V(h))
            x = x + linear2(relu(linear1(ln2(x))))

        return self.output(x)


In [ ]:
import random
import torch
import time
import ipywidgets as widgets
from IPython.display import display

mask_index = 0
alpha0 = 1.0

def sample(model, query, length, device, total_steps=20):

    out = widgets.Output()
    display(out)

    tokens = model.encoder.encode(query)
    x = torch.full((length,), mask_index, dtype=torch.long, device=device)
    x[:len(tokens)] = torch.tensor(tokens, device=device)
    clean = (x != mask_index)
    remaining = (~clean).nonzero(as_tuple=True)[0].tolist()

    with torch.no_grad():
        for step in range(total_steps):
            if not remaining:
                break

            t = 1 - step / total_steps
            s = max(0, 1 - (step + 1) / total_steps)
            alpha_t = alpha0 * (1 - t)
            alpha_s = alpha0 * (1 - s)
            p = min(1.0, max(0.0, (alpha_s - alpha_t) / max(1e-6, 1 - alpha_t)))
            n = len(remaining) if step == total_steps - 1 else max(1, int(torch.distributions.Binomial(len(remaining), torch.tensor(p)).sample().item()))
            chosen = random.sample(remaining, min(n, len(remaining)))

            clean_ids = clean.nonzero(as_tuple=True)[0]
            chosen_ids = torch.tensor(chosen, dtype=torch.long, device=device)
            pos_ids = torch.cat([clean_ids, chosen_ids])
            logits = model(x[pos_ids], pos_ids)
            logits[..., mask_index] = -float("inf")
            probs = torch.softmax(logits, dim=-1)

            for pos in chosen:
                row = (pos_ids == pos).nonzero(as_tuple=True)[0].item()
                x[pos] = torch.multinomial(probs[row], 1).item()
                clean[pos] = True
                remaining.remove(pos)

            with out:
                out.clear_output(wait=True)
                print(''.join(model.encoder.decode(x.tolist())))


## Training

1. Sample a chunk of text data: $x_0 \sim$ TinyShakespeare.
2. Sample diffusion time $t \sim Uniform(0,1)$ and mask tokens with probability $1 - \alpha_0(1 - t)$.
3. Reorder each sequence so clean tokens come first and masked tokens follow in random denoising order.
4. Run a causal transformer with the original position ids preserved.
5. Train masked positions with the weighted masked-token diffusion loss.
6. Backprop.


In [ ]:
import math
import torch
import random
import time
from tqdm import tqdm

vocab_size = len(encoder.dictionary)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open("tinyshakespeare.txt") as f:
    text = f.read()

split = int(0.9 * len(text))
train_text, val_text = text[:split], text[split:]

seq_len = 128
batch_size = 64
iterations = 100000
checkpoint_every = 10000
alpha0 = 1.0

transformer = Transformer().to(device)
optimizer = torch.optim.Adam(transformer.parameters(), lr=1e-4)

def grab_chunk(src) -> torch.Tensor:
    start = random.randint(0, len(src) - seq_len)
    return torch.tensor(transformer.encoder.encode(src[start:start + seq_len]), device=device)

def add_noise(x, t):
    alpha_t = alpha0 * (1 - t)
    mask = torch.rand_like(x, dtype=torch.float) > alpha_t
    return x.masked_fill(mask, mask_index), mask

def eso_order(zt):
    order = torch.empty_like(zt)
    for b in range(zt.shape[0]):
        clean = (zt[b] != mask_index).nonzero(as_tuple=True)[0]
        masked = (zt[b] == mask_index).nonzero(as_tuple=True)[0]
        masked = masked[torch.randperm(masked.numel(), device=zt.device)]
        order[b] = torch.cat([clean, masked])
    return order


In [ ]:
for i in tqdm(range(iterations)):
    batch = torch.stack([grab_chunk(train_text) for _ in range(batch_size)])
    t = random.uniform(1e-3, 1)
    noised, mask = add_noise(batch, t)
    if not mask.any():
        continue

    order = eso_order(noised)
    noised = noised.gather(1, order)
    targets = batch.gather(1, order)
    mask = mask.gather(1, order)

    preds = transformer(noised, order)
    preds[..., mask_index] = preds[..., mask_index].masked_fill(mask, -float("inf"))
    loss = torch.nn.functional.cross_entropy(preds[mask], targets[mask]) / t

    if i % checkpoint_every == 0:
        sample(transformer, "To be, ", 64, device)
    
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

# Save final model
sample(transformer, "To be, ", 64, device)
torch.save(transformer.state_dict(), f"eso.pt")


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transformer = Transformer().to(device)
transformer.load_state_dict(torch.load("eso.pt", map_location=device))


In [ ]:
sample(transformer, "To be, ", 128, device, 1000)


In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())

count_params(transformer)
